# Support Vector Machine

In [13]:
import os
os.makedirs("results", exist_ok=True)

import numpy as np, pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.calibration import CalibratedClassifierCV

---

In [9]:
expression = pd.read_csv("../assignment3/shared_expression_log2.csv", index_col=0)
sample_meta = pd.read_csv("../assignment3/shared_sample_metadata.csv", index_col=0)
gmm = pd.read_csv("../assignment3/results/gmm_cluster_assignments.csv", index_col=0)

assert (expression.index == sample_meta.index).all() and (gmm.index == sample_meta.index).all()

In [10]:
ranked = expression.var(axis=0).sort_values(ascending=False).index

folds = pd.read_csv("results/cv_folds.csv", index_col=0)["fold"]
cv = [(np.where(folds != i)[0], np.where(folds == i)[0]) for i in range(4)]

In [11]:
y_cond = (sample_meta["condition"] == "flight").astype(int)
y_clust = (gmm["gmm_k_2"] == sorted(gmm["gmm_k_2"].unique())[1]).astype(int)
targets = {"condition": y_cond, "cluster_k2": y_clust}

---

In [14]:
def run_svm(n_genes, y):
    X = expression[ranked[:n_genes]]
    model = make_pipeline(StandardScaler(), CalibratedClassifierCV(SVC(kernel="linear"), ensemble=False))
    proba = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]
    pred = (proba >= 0.5).astype(int)
    auc = roc_auc_score(y, proba)
    return pred, proba, auc

rows, preds, scores = [], {}, {}
for n in [10, 100, 1000, 5000, 10000]:
    for name, y in targets.items():
        pred, proba, auc = run_svm(n, y)
        rows.append({"Method": "SVM", "Genes": n, "Target": name,
                     "AUC": auc, "Accuracy": (pred == y.values).mean()})
        preds[f"svm_{name}_{n}genes"] = pred
        scores[f"svm_{name}_{n}genes"] = proba

auc_table = pd.DataFrame(rows)
auc_table.to_csv("results/svm_auc_by_gene_count.csv", index=False)
pd.DataFrame(preds, index=expression.index).to_csv("results/svm_predictions.csv")
pd.DataFrame(scores, index=expression.index).to_csv("results/svm_probabilities.csv")
auc_table

,Method,Genes,Target,AUC,Accuracy
0,SVM,10,condition,0.382812,0.50000
1,SVM,10,cluster_k2,0.976471,0.96875
2,SVM,100,condition,0.925781,0.84375
3,SVM,100,cluster_k2,1.000000,0.96875
4,SVM,1000,condition,1.000000,1.00000
5,SVM,1000,cluster_k2,0.996078,0.96875
6,SVM,5000,condition,1.000000,1.00000
7,SVM,5000,cluster_k2,0.996078,0.96875
8,SVM,10000,condition,1.000000,1.00000
9,SVM,10000,cluster_k2,0.996078,0.96875
